# UniClubs — Hybrid Recommendation Model Training

Trains a TFRS two-tower model and saves 4 sub-models for hybrid inference.

Architecture:
- `user_collab`: user_id string → learned embedding (collaborative)
- `user_content`: 12-dim feature vector → content embedding
- `event_collab`: event_id string → learned embedding (collaborative)
- `event_content`: 13-dim feature vector → content embedding

At inference: `Score = α × ContentScore + β × CollaborativeScore`

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import json
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_recommenders as tfrs

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
print('TF:', tf.__version__, '| TFRS:', tfrs.__version__)

ImportError: Package tensorflow_recommenders requires tf.keras to be Keras version 2 but got version 3.14.0. For open-source TensorFlow 2.16 and above, set the environment variable TF_USE_LEGACY_KERAS=1 to fix. For more information, see for example https://github.com/tensorflow/gnn/blob/main/tensorflow_gnn/docs/guide/keras_version.md

## 1. Constants & Vocabulary

In [ ]:
EMBEDDING_DIM   = 32
EPOCHS          = 20
BATCH_SIZE      = 64

CATEGORIES = ['tech', 'sports', 'arts', 'academic', 'social']
ALL_TAGS   = ['workshop', 'seminar', 'competition', 'social', 'networking', 'guest_speaker']
INTERESTS  = ['AI', 'Mobile Dev', 'Coding', 'Sports', 'Design', 'Art', 'Photography', 'Music', 'Academic']
INTEREST_TO_CATEGORY = {
    'AI': 'tech', 'Mobile Dev': 'tech', 'Coding': 'tech',
    'Sports': 'sports',
    'Design': 'arts', 'Art': 'arts', 'Photography': 'arts', 'Music': 'arts',
    'Academic': 'academic',
}

# Dimensions:
# user_content_vector = interest_vec(5) + attended_vec(5) + participation_freq(1) + account_age(1) = 12
# event_content_vector = category_onehot(5) + tag_vec(6) + time_of_day(1) + day_of_week(1) = 13
USER_CONTENT_DIM = 12
EVENT_CONTENT_DIM = 13

OUTPUT_DIR = '../ml_models/recommendation_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Output dir:', OUTPUT_DIR)

## 2. Synthetic Data Generation

Simulates realistic user-event interactions based on interest-category alignment.
Replace this with real Firestore data once enough interactions are collected.

In [ ]:
N_USERS        = 150
N_EVENTS       = 250
N_INTERACTIONS = 600

users = [
    {
        'user_id': f'user_{i:04d}',
        'interests': random.sample(INTERESTS, random.randint(1, 4)),
        'account_age_days': random.randint(10, 730),
    }
    for i in range(N_USERS)
]

events = [
    {
        'event_id': f'event_{i:04d}',
        'category': random.choice(CATEGORIES),
        'tags': random.sample(ALL_TAGS, random.randint(1, 3)),
        'time_of_day': random.randint(8, 20),
        'day_of_week': random.randint(0, 6),
    }
    for i in range(N_EVENTS)
]

user_dict  = {u['user_id']: u for u in users}
event_dict = {e['event_id']: e for e in events}

# 70% of interactions are biased toward user's interest categories
interactions = []
for _ in range(N_INTERACTIONS):
    user = random.choice(users)
    preferred = {INTEREST_TO_CATEGORY.get(i) for i in user['interests'] if INTEREST_TO_CATEGORY.get(i)}
    matching  = [e for e in events if e['category'] in preferred]
    event = random.choice(matching if matching and random.random() < 0.7 else events)
    interactions.append({'user_id': user['user_id'], 'event_id': event['event_id']})

interactions_df = pd.DataFrame(interactions).drop_duplicates()
print(f'Users: {len(users)}, Events: {len(events)}, Interactions: {len(interactions_df)}')

## 3. Feature Builders

Matches the feature engineering described in CLAUDE.md:
- **User features**: interests (multi-hot over categories), attended categories (multi-hot), participation frequency, account age
- **Event features**: category (one-hot), tags (multi-hot), time of day, day of week

In [ ]:
def build_user_content_vector(user, attended_categories, participation_count):
    """12-dim float32 vector: interest_vec(5) + attended_vec(5) + participation(1) + age(1)"""
    interest_vec = np.zeros(len(CATEGORIES), dtype=np.float32)
    for interest in user['interests']:
        cat = INTEREST_TO_CATEGORY.get(interest)
        if cat in CATEGORIES:
            interest_vec[CATEGORIES.index(cat)] = 1.0

    attended_vec = np.zeros(len(CATEGORIES), dtype=np.float32)
    for cat in attended_categories:
        if cat in CATEGORIES:
            attended_vec[CATEGORIES.index(cat)] = 1.0

    participation_freq = np.array([min(participation_count / 20.0, 1.0)], dtype=np.float32)
    account_age        = np.array([min(user['account_age_days'] / 730.0, 1.0)], dtype=np.float32)
    return np.concatenate([interest_vec, attended_vec, participation_freq, account_age])


def build_event_content_vector(event):
    """13-dim float32 vector: category_onehot(5) + tag_vec(6) + time(1) + day(1)"""
    cat_onehot = np.zeros(len(CATEGORIES), dtype=np.float32)
    if event['category'] in CATEGORIES:
        cat_onehot[CATEGORIES.index(event['category'])] = 1.0

    tag_vec = np.zeros(len(ALL_TAGS), dtype=np.float32)
    for tag in event.get('tags', []):
        if tag in ALL_TAGS:
            tag_vec[ALL_TAGS.index(tag)] = 1.0

    time_of_day  = np.array([event['time_of_day'] / 24.0], dtype=np.float32)
    day_of_week  = np.array([event['day_of_week'] / 7.0],  dtype=np.float32)
    return np.concatenate([cat_onehot, tag_vec, time_of_day, day_of_week])

## 4. Build the 4 Sub-Models

In [ ]:
ALL_USER_IDS  = [u['user_id'] for u in users]
ALL_EVENT_IDS = [e['event_id'] for e in events]

# 1. User Collaborative: user_id → EMBEDDING_DIM
uid_in       = tf.keras.Input(shape=(), dtype=tf.string, name='user_id')
uid_lookup   = tf.keras.layers.StringLookup(vocabulary=ALL_USER_IDS, mask_token=None)(uid_in)
uid_emb      = tf.keras.layers.Embedding(len(ALL_USER_IDS) + 1, EMBEDDING_DIM)(uid_lookup)
uid_out      = tf.keras.layers.Dense(EMBEDDING_DIM, name='user_collab_out')(uid_emb)
user_collab_model = tf.keras.Model(inputs=uid_in, outputs=uid_out, name='user_collab')

# 2. User Content: 12-dim vector → EMBEDDING_DIM
ucontent_in  = tf.keras.Input(shape=(USER_CONTENT_DIM,), dtype=tf.float32, name='user_content')
ucontent_h   = tf.keras.layers.Dense(EMBEDDING_DIM, activation='relu')(ucontent_in)
ucontent_out = tf.keras.layers.Dense(EMBEDDING_DIM, name='user_content_out')(ucontent_h)
user_content_model = tf.keras.Model(inputs=ucontent_in, outputs=ucontent_out, name='user_content')

# 3. Event Collaborative: event_id → EMBEDDING_DIM
eid_in       = tf.keras.Input(shape=(), dtype=tf.string, name='event_id')
eid_lookup   = tf.keras.layers.StringLookup(vocabulary=ALL_EVENT_IDS, mask_token=None)(eid_in)
eid_emb      = tf.keras.layers.Embedding(len(ALL_EVENT_IDS) + 1, EMBEDDING_DIM)(eid_lookup)
eid_out      = tf.keras.layers.Dense(EMBEDDING_DIM, name='event_collab_out')(eid_emb)
event_collab_model = tf.keras.Model(inputs=eid_in, outputs=eid_out, name='event_collab')

# 4. Event Content: 13-dim vector → EMBEDDING_DIM
econtent_in  = tf.keras.Input(shape=(EVENT_CONTENT_DIM,), dtype=tf.float32, name='event_content')
econtent_h   = tf.keras.layers.Dense(EMBEDDING_DIM, activation='relu')(econtent_in)
econtent_out = tf.keras.layers.Dense(EMBEDDING_DIM, name='event_content_out')(econtent_h)
event_content_model = tf.keras.Model(inputs=econtent_in, outputs=econtent_out, name='event_content')

print('4 sub-models built successfully.')

## 5. TFRS Training Model

In [ ]:
class HybridRecommender(tfrs.Model):
    """Trains all 4 sub-models jointly via the TFRS Retrieval task."""

    def __init__(self, user_collab, user_content, event_collab, event_content, candidates_ds):
        super().__init__()
        self.user_collab   = user_collab
        self.user_content  = user_content
        self.event_collab  = event_collab
        self.event_content = event_content

        self.task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(
                candidates=candidates_ds.batch(128).map(self._event_embedding)
            )
        )

    def _user_embedding(self, features):
        collab  = self.user_collab(features['user_id'])
        content = self.user_content(features['user_content_vector'])
        return tf.concat([collab, content], axis=-1)

    def _event_embedding(self, features):
        collab  = self.event_collab(features['event_id'])
        content = self.event_content(features['event_content_vector'])
        return tf.concat([collab, content], axis=-1)

    def compute_loss(self, features, training=False):
        return self.task(
            self._user_embedding(features),
            self._event_embedding(features),
            compute_metrics=not training,
        )

## 6. Prepare TF Dataset

In [ ]:
# Build per-user attendance history from interactions
user_history = {}
for _, row in interactions_df.iterrows():
    uid, eid = row['user_id'], row['event_id']
    cat = event_dict[eid]['category']
    if uid not in user_history:
        user_history[uid] = {'categories': [], 'count': 0}
    user_history[uid]['categories'].append(cat)
    user_history[uid]['count'] += 1

# Build training tensors
t_user_ids, t_event_ids, t_user_vecs, t_event_vecs = [], [], [], []
for _, row in interactions_df.iterrows():
    user    = user_dict[row['user_id']]
    event   = event_dict[row['event_id']]
    history = user_history.get(row['user_id'], {'categories': [], 'count': 0})
    t_user_ids.append(row['user_id'])
    t_event_ids.append(row['event_id'])
    t_user_vecs.append(build_user_content_vector(user, history['categories'], history['count']))
    t_event_vecs.append(build_event_content_vector(event))

train_ds = tf.data.Dataset.from_tensor_slices({
    'user_id':              t_user_ids,
    'user_content_vector':  np.array(t_user_vecs),
    'event_id':             t_event_ids,
    'event_content_vector': np.array(t_event_vecs),
}).shuffle(1000).batch(BATCH_SIZE)

# Candidate events dataset for FactorizedTopK
all_events_ds = tf.data.Dataset.from_tensor_slices({
    'event_id':             [e['event_id'] for e in events],
    'event_content_vector': np.array([build_event_content_vector(e) for e in events]),
})

print(f'Training samples: {len(t_user_ids)}')

## 7. Train

In [ ]:
model = HybridRecommender(
    user_collab_model, user_content_model,
    event_collab_model, event_content_model,
    all_events_ds,
)
model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))
model.fit(train_ds, epochs=EPOCHS)
print('Training complete.')

## 8. Save Models & Vocabularies

In [ ]:
user_collab_model.export(os.path.join(OUTPUT_DIR, 'user_collab'))
user_content_model.export(os.path.join(OUTPUT_DIR, 'user_content'))
event_collab_model.export(os.path.join(OUTPUT_DIR, 'event_collab'))
event_content_model.export(os.path.join(OUTPUT_DIR, 'event_content'))

with open(os.path.join(OUTPUT_DIR, 'vocabularies.json'), 'w') as f:
    json.dump({
        'categories':           CATEGORIES,
        'all_tags':             ALL_TAGS,
        'interests':            INTERESTS,
        'interest_to_category': INTEREST_TO_CATEGORY,
        'user_content_dim':     USER_CONTENT_DIM,
        'event_content_dim':    EVENT_CONTENT_DIM,
    }, f, indent=2)

print('Saved to:', OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

## 9. Inference Test

Verifies the saved models load and score correctly before wiring into FastAPI.

In [ ]:
# Load saved models
u_collab  = tf.saved_model.load(os.path.join(OUTPUT_DIR, 'user_collab'))
u_content = tf.saved_model.load(os.path.join(OUTPUT_DIR, 'user_content'))
e_collab  = tf.saved_model.load(os.path.join(OUTPUT_DIR, 'event_collab'))
e_content = tf.saved_model.load(os.path.join(OUTPUT_DIR, 'event_content'))

def infer(u_collab, u_content, e_collab, e_content, user, event, history, alpha, beta):
    u_cv = build_user_content_vector(user, history['categories'], history['count'])
    e_cv = build_event_content_vector(event)

    uc_emb  = list(u_collab.signatures['serving_default'](user_id=tf.constant([user['user_id']])).values())[0]
    uco_emb = list(u_content.signatures['serving_default'](user_content=tf.constant([u_cv])).values())[0]
    ec_emb  = list(e_collab.signatures['serving_default'](event_id=tf.constant([event['event_id']])).values())[0]
    eco_emb = list(e_content.signatures['serving_default'](event_content=tf.constant([e_cv])).values())[0]

    collab_score  = float(tf.reduce_sum(uc_emb * ec_emb))
    content_score = float(tf.reduce_sum(uco_emb * eco_emb))
    return alpha * content_score + beta * collab_score

test_user  = users[0]
test_event = events[0]
history    = user_history.get(test_user['user_id'], {'categories': [], 'count': 0})
score = infer(u_collab, u_content, e_collab, e_content, test_user, test_event, history, 0.4, 0.6)

print(f'User: {test_user["user_id"]} | Interests: {test_user["interests"]}')
print(f'Event: {test_event["event_id"]} | Category: {test_event["category"]}')
print(f'Hybrid score: {score:.4f}')
print('Inference test passed!')